# Day 74 — Loss Functions

This notebook covers Loss vs. Cost Functions, Regression Loss Functions (MSE, MAE), Classification Loss Functions (Binary Cross-Entropy), Differentiability for Gradient Descent, and Numerical Stability (`np.clip`)

## Validation goals
- Distinguish between sample-level loss and dataset-level cost as optimization objectives
- Analyze regression loss functions (MSE and MAE) and compare linear vs. quadratic error scaling
- Understand Binary Cross-Entropy (BCE) for probabilistic classification and why confident misclassifications incur heavy logarithmic penalties
- Implement MSE and BCE from scratch in Python using NumPy with epsilon clipping for numerical stability

### 1. What is a Loss Function?

A Loss Function (or cost function) is a mathematical formula that quantifies the penalty or error between your model's prediction ($\hat{y}$) and the true target label ($y$).

- Loss vs. Cost: Loss refers to the error computed for a single data sample, while Cost is the average loss across an entire batch or dataset.
- The Optimization Goal: During training, the neural network adjusts its weights and biases to minimize this loss value. If the loss function is zero, the model's predictions match the ground truth perfectly.

### 2. Core Loss Functions for Regression

When your neural network predicts a continuous numerical value (such as housing prices or our previous Nigeria Retail Demand Predictor), you use regression loss functions.

**A. Mean Squared Error (MSE)**

$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$
- Behavior: Squares the errors. Because errors are squared, large mistakes incur heavy penalties, making the gradient steep when predictions are far off.
- Use Case: Standard default for regression tasks.

**B. Mean Absolute Error (MAE)**

$$\text{MAE} = \frac{1}{n} \sum_{i=1}^{n} \vert{}y_i - \hat{y}_i$$
- Behavior: Measures the absolute distance without squaring. It is less sensitive to outliers than MSE.
- Use Case: Regression tasks where outliers should not disproportionately dominate training.

### 3. Core Loss Functions for Classification

When your neural network outputs probabilities for discrete categories, regression loss functions fail. We need cross-entropy loss.

**A. Binary Cross-Entropy (BCE)**

For binary classification ($y \in \{0, 1\}$), where $\hat{y}$ is the predicted probability of the positive class:$$\text{BCE} = -\frac{1}{n} \sum_{i=1}^{n} \left[ y_i \log(\hat{y}_i) + (1 - y_i) \log(1 - \hat{y}_i) \right]$$
- Behavior: Heavily penalizes confident, incorrect predictions. If the true label is 1 and the model predicts 0.01 with high confidence, $\log(0.01)$ becomes a massive negative number, driving the loss up steeply.

### 4. Implementing Loss Functions from Scratch in Python (NumPy)

Let's write a clean Python script implementing Mean Squared Error and Binary Cross-Entropy using NumPy.

In [1]:
import numpy as np

# 1. Regression Loss: Mean Squared Error (MSE)
def compute_mse(y_true, y_pred):
    """
    Computes Mean Squared Error between true targets and predictions.
    """
    return np.mean((y_true - y_pred) ** 2)

# 2. Classification Loss: Binary Cross-Entropy (BCE)
def compute_binary_cross_entropy(y_true, y_pred):
    """
    Computes Binary Cross-Entropy loss.
    Adds a small epsilon (1e-15) to prevent log(0) calculation errors.
    """
    epsilon = 1e-15
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
    
    bce = -np.mean(
        y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred)
    )
    return bce

# Example Execution
# Regression Test
true_sales = np.array([120.0, 45.0, 300.0, 150.0])
predicted_sales = np.array([110.0, 50.0, 280.0, 155.0])
mse_loss = compute_mse(true_sales, predicted_sales)

# Binary Classification Test (Spam Detection)
true_labels = np.array([1.0, 0.0, 1.0, 0.0])
predicted_probs = np.array([0.85, 0.12, 0.90, 0.25])
bce_loss = compute_binary_cross_entropy(true_labels, predicted_probs)

print("--- Day 74: Loss Functions Demonstration ---")
print(f"Computed MSE Loss: {mse_loss:.4f}")
print(f"Computed BCE Loss: {bce_loss:.4f}")

--- Day 74: Loss Functions Demonstration ---
Computed MSE Loss: 137.5000
Computed BCE Loss: 0.1708


### Guided Question:

Why is a loss function mathematically required for training a neural network via gradient descent, rather than simply measuring evaluation metrics like accuracy or $R^2$ score?

### Answer:

A loss function is mathematically required because gradient descent relies on smooth, continuous gradients (slopes) to figure out how to adjust the network's weights, whereas evaluation metrics like accuracy or $R^2$ cannot provide this directional guidance.

The distinction becomes clear through two main factors:

- Gradient Descent Needs a "Slope": Gradient descent works like walking down a foggy hill—it needs to feel the slope under its feet to know which direction is downhill. Loss functions are specially designed to output a continuous, smooth measure of error for every tiny change in weights.

- **Metrics are "Dead Ends":**

    - Accuracy is a step function (a prediction is either right or wrong). If your model improves slightly, its accuracy might still be stuck at 85%. Because accuracy doesn't change smoothly, its derivative (slope) is zero almost everywhere, giving gradient descent zero information on how to update the weights.
    - $R^2$ Score (and accuracy) are final report cards used to evaluate performance after the fact. They tell you how well the model did overall, but they cannot act as the compass that guides step-by-step learning.

### Guided Question:

What is the primary mathematical difference in how Mean Squared Error (MSE) and Mean Absolute Error (MAE) treat large prediction errors?

### Answer:

The primary mathematical difference lies in how they scale the penalty: MAE uses a linear penalty, while MSE uses a quadratic (squared) penalty.

**Mean Absolute Error (MAE):**

Calculates the absolute distance between predictions and actual values:$$\text{MAE} = \vert{}y - \hat{y}\vert{}$$
- Effect: The penalty grows at a constant, steady rate. If your error doubles from 2 to 4, the penalty also simply doubles from 2 to 4.

**Mean Squared Error (MSE):**

 Squares the distance between predictions and actual values:$$\text{MSE} = (y - \hat{y})^2$$
- Effect: The penalty grows exponentially faster the larger the error gets. If your error doubles from 2 to 4, the penalty quadruples from $2^2 = 4$ to $4^2 = 16$.

### Guided Question:

In the Binary Cross-Entropy implementation above, why do we clip the predicted probabilities using `np.clip(y_pred, epsilon, 1 - epsilon)` before computing the logarithm?

### Answer:

We clip the predicted probabilities with `np.clip` to prevent mathematical crashes and NaN (Not a Number) errors caused by taking the logarithm of zero.

Here is why it happens:

- The Problem with $\log(0)$: In mathematics, the natural logarithm of zero ($\ln(0)$) is undefined (it approaches negative infinity, $-\infty$). If your model makes a confident mistake—such as predicting a probability of exactly 0.0 for a true label of 1—the code will attempt to calculate $\log(0)$, causing a runtime error or turning your loss value into NaN.

- The Solution (Clipping): `np.clip(y_pred, epsilon, 1 - epsilon)` forces every prediction to stay within a tiny, safe boundary (between `0.000000000000001` and `0.999999999999999`). This ensures the log function never receives a true 0 or 1, keeping your calculations mathematically stable without crashing your program.